<a href="https://colab.research.google.com/github/valliansayoga/ey-data-challenge-2025/blob/master/EY2025_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
scl_mapping = {
    0: "No Data".lower().replace(" ", "_"),
    1: "Saturated or defective pixel".lower().replace(" ", "_"),
    2: "Topographic casted shadows".lower().replace(" ", "_"),
    3: "Cloud shadows".lower().replace(" ", "_"),
    4: "Vegetation".lower().replace(" ", "_"),
    5: "Not vegetated".lower().replace(" ", "_"),
    6: "Water".lower().replace(" ", "_"),
    7: "Unclassified".lower().replace(" ", "_"),
    8: "Cloud medium probability".lower().replace(" ", "_"),
    9: "Cloud high probability".lower().replace(" ", "_"),
    10: "Thin cirrus".lower().replace(" ", "_"),
    11: "Snow or ice".lower().replace(" ", "_"),

}
scl_mapping

{0: 'no_data',
 1: 'saturated_or_defective_pixel',
 2: 'topographic_casted_shadows',
 3: 'cloud_shadows',
 4: 'vegetation',
 5: 'not_vegetated',
 6: 'water',
 7: 'unclassified',
 8: 'cloud_medium_probability',
 9: 'cloud_high_probability',
 10: 'thin_cirrus',
 11: 'snow_or_ice'}

In [10]:
from sklearn.preprocessing import OneHotEncoder

to_drop = ["Latitude", "Longitude", "datetime",
        #    "is_a_building",
        #    "relative_position",
        #    "aot_median",
        #    "scl_median",
        #    "ndvi_median"
           ]
target = "UHI Index"

df = pd.read_csv("Train_Final.csv").drop(to_drop, axis=1, errors="ignore")

# # Uncomment if used
# to_drop = np.concatenate((to_drop, df.columns[df.columns.str.contains("count", regex=True)]))
# df.drop(columns=to_drop, inplace=True, errors="ignore")

# # Experiment feature engineering
# NDVI x LST
df["evi x lwir"] = df.evi_median * df.lwir_median
# NDBI x LST
df["ndbi x lwir"] = df.ndbi_median * df.lwir_median

# Experiment using top 80% features
use_cols = [
    target,
    "std_distance",
    "average_distance",
    "distance_range",
    "distance_variation",
    "lwir_median",
    "wvp_median",
    "emsd",
    "neighboring_intersection",
    "nearest_building_size",
    "avg_polygon_complexity",
    "evi_median",
    "coast_aerosol_median",
    "ndwi_median",
    "building_area_density",
    "evi x lwir",
    "ndvi_median",
    "nir_median",
]
df = df.loc[:, use_cols]

# # Comment if not used!
# ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
# df.scl_median = df.scl_median.map(scl_mapping)
# scl_ohe = ohe.fit_transform(df.loc[:, ["scl_median"]])
# scl_ohe = pd.DataFrame(scl_ohe, columns=ohe.get_feature_names_out(["scl_median"]))
# df = pd.concat([df.drop("scl_median", axis=1), scl_ohe], axis=1)

In [11]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


def create_train(df_features, scaler, train_size=0.8, indices=None):
    print("Removing duplicates...")
    rows_before = df_features.shape[0]
    check_dupl = df_features.columns[1:]
    df_features = df_features.drop_duplicates(subset=check_dupl, keep='first')
    rows_after = df_features.shape[0]
    print(f"Removed {rows_before-rows_after} duplicate rows!")

    X = df_features.drop(target, axis=1)
    y = df_features[target]

    print("Scaling...")
    if indices is not None:
        X_train, X_test, y_train, y_test = X.iloc[indices[0]], X.iloc[indices[1]], y.iloc[indices[0]], y.iloc[indices[1]]
    else:
        X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0, train_size=train_size)

    X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
    X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)
    print("Done")
    return X_train, X_test, y_train, y_test, scaler
scaler = StandardScaler()

X_train, X_test, y_train, y_test, scaler = create_train(
    df,
    scaler,
    # indices=(train_index, test_index)
)

Removing duplicates...
Removed 0 duplicate rows!
Scaling...
Done


In [14]:
# from sklearn.feature_selection import SelectKBest, f_regression, SequentialFeatureSelector
# from sklearn.ensemble import ExtraTreesRegressor
# from sklearn.metrics import r2_score

# estimator = ExtraTreesRegressor(n_estimators=200, random_state=0, n_jobs=-1)
# selector = SequentialFeatureSelector(cv=2, estimator=estimator, n_jobs=-1)
# selector.fit(X_train, y_train)
# X_train = pd.DataFrame(selector.transform(X_train), columns=X_train.columns[selector.get_support()])
# X_test = pd.DataFrame(selector.transform(X_test), columns=X_test.columns[selector.get_support()])

# Modelling

In [5]:
from sklearn.metrics import r2_score


def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    insample = r2_score(y_train, model.predict(X_train))
    outsample = r2_score(y_test, model.predict(X_test))
    return insample, outsample

In [12]:
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from tqdm import tqdm
models = [
    # {"model": RandomForestRegressor(random_state=0, n_jobs=-1)},
    # # {"model": RandomForestRegressor(200, random_state=0, n_jobs=-1)},
    # {"model": RandomForestRegressor(250, random_state=0, n_jobs=-1)},
    # # {"model": RandomForestRegressor(300, random_state=0, n_jobs=-1)},
    # {"model": RandomForestRegressor(150, random_state=0, n_jobs=-1)},
    # {"model": ExtraTreesRegressor(n_estimators=200, random_state=0, n_jobs=-1, bootstrap=True, oob_score=True)},
    {"model": ExtraTreesRegressor(n_estimators=200, random_state=0, n_jobs=-1)},
    # # {"model": ExtraTreesRegressor(n_estimators=250, random_state=0, n_jobs=-1, bootstrap=True, oob_score=True)},
    {"model": ExtraTreesRegressor(n_estimators=250, random_state=0, n_jobs=-1)},
    # {"model": ExtraTreesRegressor(n_estimators=300, random_state=0, n_jobs=-1)},
    {"model": ExtraTreesRegressor(n_estimators=100, random_state=0, n_jobs=-1)},
    # {"model": KNeighborsRegressor(3, n_jobs=-1)},
    # {"model": KNeighborsRegressor(5, n_jobs=-1)},
    # {"model": KNeighborsRegressor(7, n_jobs=-1)},
    # {"model": DecisionTreeRegressor(random_state=0)},
    # {"model": MLPRegressor((32, 64, 128), random_state=0, early_stopping=True, learning_rate_init=2.5e-4)},
]

for model in tqdm(models):
    insample, outsample = evaluate_model(model["model"], X_train, X_test, y_train, y_test)
    model["insample"] = insample
    model["outsample"] = outsample

results = pd.DataFrame(models).sort_values("outsample", ascending=False).reset_index(drop=True)
results
# 1.000000	0.934180 | 1.000000	0.942341

100%|██████████| 3/3 [00:27<00:00,  9.05s/it]


,model,insample,outsample
0,"(ExtraTreeRegressor(random_state=209652396), E...",1.0,0.959951
1,"(ExtraTreeRegressor(random_state=209652396), E...",1.0,0.959936
2,"(ExtraTreeRegressor(random_state=209652396), E...",1.0,0.959699


In [15]:
best_model = results.iloc[0]
best_model.model

ExtraTreesRegressor(n_estimators=200, n_jobs=-1, random_state=0)

In [16]:
importance = pd.DataFrame(
    {"Features": X_train.columns, "Importance": best_model.model.feature_importances_},
).sort_values("Importance", ascending=False).reset_index(drop=True)
importance["cumulative_importance"] = importance.Importance.cumsum() / importance.Importance.sum()
importance

,Features,Importance,cumulative_importance
0,distance_range,0.137792,0.137792
1,average_distance,0.135152,0.272944
2,std_distance,0.131622,0.404566
3,distance_variation,0.131438,0.536004
4,lwir_median,0.050157,0.586161
5,emsd,0.044480,0.630640
6,wvp_median,0.042969,0.673610
7,building_area_density,0.039516,0.713126
8,coast_aerosol_median,0.035925,0.749051
9,neighboring_intersection,0.035096,0.784147


# Predicting Submission

In [17]:
def create_submission(filename: str, model, scaler):
    sub_df = pd.read_csv("Submission_Final.csv")
    final_df = sub_df[["Latitude", "Longitude"]].copy()
    print("Predicting", sub_df.shape[0], "rows...")

    ############################
    sub_df["evi x lwir"] = sub_df.evi_median * sub_df.lwir_median
    # NDBI x LST
    sub_df["ndbi x lwir"] = sub_df.ndbi_median * sub_df.lwir_median

    # # # # # Comment if not used!
    # sub_df.scl_median = sub_df.scl_median.map(scl_mapping)
    # scl_ohe = ohe.transform(sub_df.loc[:, ["scl_median"]])
    # scl_ohe = pd.DataFrame(scl_ohe, columns=ohe.get_feature_names_out(["scl_median"]))
    # sub_df = pd.concat([sub_df.drop("scl_median", axis=1), scl_ohe], axis=1)

    to_predict = pd.DataFrame(
        scaler.transform(sub_df.loc[:, X_train.columns]),
        columns=X_train.columns
    )

    print("Predicting...")
    final_df["UHI Index"] = model.predict(to_predict)
    final_df.to_csv(filename, index=False)
    print("Done!")
    return
create_submission("BestModel_Radius50_Pareto.csv", best_model.model, scaler)

Predicting 1040 rows...
Predicting...
Done!


---